# BITS Pilani - Work Integrated Learning Programmes (WILP)
## M.Tech (AIML) - Machine Learning (Assignment 2)
### Classification Modeling, Evaluation & Streamlit Deployment

**Dataset:** Online Shoppers Purchasing Intention (UCI Machine Learning Repository)  
**Student ID:** 2026AJ05386

---

### Assignment Objectives
1. Select a dataset with ≥ 12 features and ≥ 500 instances from UCI / Kaggle.
2. Train and evaluate 5 ML Classifiers: Logistic Regression, Decision Tree, KNN, Naive Bayes, SVM.
3. Compute 6 evaluation metrics: Accuracy, ROC-AUC, Precision, Recall, F1-Score, MCC.
4. Save trained models and deploy via Streamlit.


### 1. Import Required Libraries

In [ ]:
import os
import urllib.request
import zipfile
import io
import warnings
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, roc_auc_score, precision_score,
    recall_score, f1_score, matthews_corrcoef,
    confusion_matrix, classification_report
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
print("All libraries imported successfully.")


### 2. Dataset Acquisition & Exploratory Data Analysis (EDA)

In [ ]:
# Download from UCI Repository
URL = 'https://archive.ics.uci.edu/static/public/468/online+shoppers+purchasing+intention+dataset.zip'
req = urllib.request.Request(URL, headers={'User-Agent': 'Mozilla/5.0'})
with urllib.request.urlopen(req) as resp:
    z = zipfile.ZipFile(io.BytesIO(resp.read()))
    df = pd.read_csv(z.open('online_shoppers_intention.csv'))

print(f"Dataset Shape: {df.shape[0]} rows, {df.shape[1]} columns")
df.head()


In [ ]:
# Data types and missing values
print("Data Types:")
print(df.dtypes)
print("\nMissing Values:")
print(df.isnull().sum())
print("\nTarget Class Distribution ('Revenue'):")
print(df['Revenue'].value_counts(normalize=True))


In [ ]:
# Visualize target distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

df['Revenue'].value_counts().plot(kind='bar', ax=axes[0], color=['#3B82F6', '#EF4444'])
axes[0].set_title('Target Class Distribution', fontweight='bold')
axes[0].set_xticklabels(['No Revenue (0)', 'Revenue (1)'], rotation=0)
axes[0].set_ylabel('Count')

# Correlation heatmap of numeric features
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), cmap='coolwarm', center=0, ax=axes[1],
            fmt='.1f', linewidths=0.5, cbar_kws={'shrink': 0.8})
axes[1].set_title('Feature Correlation Heatmap', fontweight='bold')
plt.tight_layout()
plt.show()


### 3. Data Preprocessing & Train-Test Split

In [ ]:
# Convert booleans to binary integer
df['Revenue'] = df['Revenue'].astype(int)
df['Weekend'] = df['Weekend'].astype(int)

numeric_cols = [
    'Administrative', 'Administrative_Duration', 'Informational', 'Informational_Duration',
    'ProductRelated', 'ProductRelated_Duration', 'BounceRates', 'ExitRates',
    'PageValues', 'SpecialDay', 'OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Weekend'
]
categorical_cols = ['Month', 'VisitorType']
feature_cols = numeric_cols + categorical_cols
target_col = 'Revenue'

X = df[feature_cols]
y = df[target_col]

# 80-20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}, y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape},  y_test shape:  {y_test.shape}")

# Save test_data.csv for Streamlit deployment
test_export = X_test.copy()
test_export['Revenue'] = y_test
test_export.to_csv('../test_data.csv', index=False)
print("\nSaved test_data.csv successfully.")


### 4. Build Preprocessor Pipeline

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), categorical_cols)
    ]
)
preprocessor.fit(X_train)
joblib.dump(preprocessor, 'preprocessor.joblib')
print("Preprocessor fitted and saved.")


### 5. Training and Evaluating 5 Classification Models

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, solver='liblinear', random_state=42),
    'Decision Tree': DecisionTreeClassifier(max_depth=6, min_samples_leaf=10, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=7, weights='distance'),
    'Naive Bayes': GaussianNB(var_smoothing=1e-1),
    'Support Vector Machine': SVC(kernel='rbf', C=1.0, probability=True, random_state=42)
}

model_filenames = {
    'Logistic Regression': 'logistic_regression.joblib',
    'Decision Tree': 'decision_tree.joblib',
    'KNN': 'knn.joblib',
    'Naive Bayes': 'naive_bayes.joblib',
    'Support Vector Machine': 'svm.joblib'
}

metrics_list = []
cms = {}

for name, clf in models.items():
    print(f"Training {name}...")
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classifier', clf)
    ])
    pipe.fit(X_train, y_train)
    joblib.dump(pipe, model_filenames[name])

    y_pred = pipe.predict(X_test)
    y_prob = pipe.predict_proba(X_test)[:, 1]

    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    mcc = matthews_corrcoef(y_test, y_pred)

    cms[name] = confusion_matrix(y_test, y_pred)

    metrics_list.append({
        'ML Model Name': name,
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    })

comparison_df = pd.DataFrame(metrics_list)
print("\n=== MODEL COMPARISON TABLE ===")
comparison_df


### 6. Confusion Matrices for All 5 Classifiers

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (name, cm) in enumerate(cms.items()):
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=axes[idx],
                xticklabels=['No Revenue', 'Revenue'], yticklabels=['No Revenue', 'Revenue'])
    axes[idx].set_title(f"{name}", fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

# Hide unused subplot
fig.delaxes(axes[5])
plt.suptitle("Confusion Matrices — All 5 Classifiers", fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 7. Classification Reports

In [ ]:
for name, clf_file in model_filenames.items():
    pipe = joblib.load(clf_file)
    y_pred = pipe.predict(X_test)
    print(f"\n{'='*50}")
    print(f"Classification Report: {name}")
    print('='*50)
    print(classification_report(y_test, y_pred, target_names=['No Revenue', 'Revenue']))


### 8. Performance Observations & Overall Winner

| ML Model Name | Observation |
| :--- | :--- |
| **Logistic Regression** | Achieves the highest precision (74.3%) with very few false positives, but suffers from low recall (35.6%). The linear decision boundary misses complex multi-feature interaction patterns in browsing behavior. AUC is strong at 0.8878, indicating good ranking ability. |
| **Decision Tree** | **Best overall performer.** Balances precision (72.5%) and recall (52.4%) effectively. Non-linear threshold-based splits on `PageValues` and `ExitRates` directly capture hierarchical purchase-intent rules. Achieves highest ROC-AUC (0.9141), F1 (0.6079), and MCC (0.5590). |
| **KNN** | Moderate performance (Accuracy 87.2%, AUC 0.80). Suffers from the curse of dimensionality in the 17-feature session space. Recall is low (36.9%). |
| **Naive Bayes** | Highest recall (60.5%) among all models, but lower precision (43.3%) because correlated features (BounceRates, ExitRates, PageValues) violate the independence assumption. |
| **SVM** | Second-best overall with accuracy 88.65% and MCC 0.5113. RBF kernel separates non-linear clusters well, but slightly underperforms Decision Tree on AUC and recall. |

---

### Overall Winner: Decision Tree Classifier

1. **Highest ROC-AUC (0.9141)** — Best discriminative ability across all thresholds.
2. **Highest F1-Score (0.6079)** — Best precision-recall balance on the imbalanced dataset.
3. **Highest MCC (0.5590)** — Most reliable metric for imbalanced binary classification.
4. **Interpretability** — Splits can be directly explained to business stakeholders.
